# Item feature extraction

Turns the raw item text (title / description / feature bullets) into structured
per-item features, using the category schema in `data/master_metadata.json`.

The whole pipeline is six calls, and every one of them lives in
`extract_features.py` — this notebook only drives them:

```python
df_result   = run_feature_extraction(df_items, text_columns, master_metadata, global_filters, ...)
df_expanded = expand_features(df_result)          # dict -> typed columns
df_canon    = canonicalize_values(df_expanded)    # grey -> gray            (Filter 4)
df_canon    = clean_brand(df_canon)               # rare brands -> other    (Filter 5)
df_canon    = normalize_dimensions(df_canon)      # dimensions -> inches    (Filter 6)
df_clean    = clean_numeric_ranges(df_canon)      # out-of-range -> NaN     (Filter 3)
save_features(df_clean, DATA_DIR / 'df_features.pkl')
```

Filters 1 (rows without a `cat_3` schema) and 2 (marketing boilerplate) run
inside `run_feature_extraction`. See `README.md` for what each one does.

**Takes ~40 min on the full 1.13M items.** Set `nrows=` in the load cell for a
smoke pass first.

In [12]:
import sys
from pathlib import Path

# Make the package importable when running from this folder
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
from nltk.stem import WordNetLemmatizer

from feature_extraction_workflow import (
    build_global_filter_regex,
    build_stopwords,
    clean_text,
    ensure_cat_columns,
    extract_features,
    filter_by_cat_3,
    merge_extracted,
    parse_list_string,
    remove_global_filters,
    run_feature_extraction,
)

pd.options.display.max_colwidth = 80
pd.options.display.max_columns = None

## 1. Load the item table and the schema

Read a sample of items plus the schema and global-filter configs. A 50k-row sample keeps the notebook fast while still hitting most categories.

In [13]:
DATA_DIR = PROJECT_ROOT / 'data'

df_items = pd.read_csv(
    DATA_DIR / 'meta_Home_and_Kitchen_filtered.csv',
    low_memory=False,
    # nrows=50_000,
).drop_duplicates()

print(f'df_items: {len(df_items):,} rows')
df_items[['asin', 'title', 'category', 'description', 'feature']].head(3)

df_items: 1,285,392 rows


,asin,title,category,description,feature
0,0001487795,You Are Special Today Red Plate [With Red Pen],"['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Dinnerware'...",['It was a time honored tradition among the early American families that whe...,[]
1,0002020300,Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy,"['Home & Kitchen', 'Home Dcor', 'Candles & Holders', 'Candles']",['VICKS INHALER relieves stuffy noses helps sinus congestion breathe easy gr...,[]
2,0006564224,Artistic Churchware Communion Cup Filler: RW525,"['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Glassware &...","['16 oz squeeze bottle, 1 lb.']","['Religious Supply Center', 'RW-525', 'Communion Cup Filler']"


In [15]:
with open(DATA_DIR / 'master_metadata.json') as f:
    master_metadata = json.load(f)

with open(DATA_DIR / 'global_filters.json') as f:
    global_filters = json.load(f)

print(f'master_metadata categories: {len(master_metadata)}')
print(f'global_filters phrases: {len(global_filters)}')
print(f'\nExample category schema (cat_3 = {next(iter(master_metadata))!r}):')
first_cat = next(iter(master_metadata))
print(json.dumps({first_cat: master_metadata[first_cat]}, indent=2)[:500])

master_metadata categories: 69
global_filters phrases: 41

Example category schema (cat_3 = 'Bakeware'):
{
  "Bakeware": {
    "Brand": {
      "type": "dictionary",
      "values": [
        "ann clark",
        "cybrtrayd",
        "wilton",
        "fat daddio",
        "nordic ware",
        "le creuset",
        "bia cordon bleu",
        "fox run",
        "paderno world cuisine",
        "ck product",
        "villeroy & boch",
        "coppergifts",
        "first impression",
        "ateco",
        "meri meri",
        "matfer bourgeat",
        "pampered chef",
        "chicago metallic


## 2. Sanity-check the extractor on a few titles

One call on a handful of cleaned titles, to confirm the schema is being applied
before committing to the full run.

In [21]:
# extract_features on a few cleaned titles
sample = df_filtered[['asin', 'title', 'cat_3']].dropna(subset=['title', 'cat_3']).head(5).copy()
sample['title_cleaned'] = sample['title'].apply(
    lambda t: remove_global_filters(
        clean_text(t, stop_words, lemmatizer),
        filter_regex,
    )
)
sample['features'] = sample.apply(
    lambda r: extract_features(r['title_cleaned'], r['cat_3'], master_metadata),
    axis=1,
)

for _, row in sample.iterrows():
    print(f"cat_3   : {row['cat_3']}")
    print(f"title   : {row['title'][:80]}")
    print(f"cleaned : {row['title_cleaned'][:80]}")
    print(f"features: {row['features']}")
    print()

cat_3   : Dining & Entertaining
title   : You Are Special Today Red Plate [With Red Pen]
cleaned : special today red plate red pen
features: {'Color': 'red'}

cat_3   : Candles & Holders
title   : Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy
cleaned : vicks inhaler relief cold sinus nasal congestion allergy
features: {}

cat_3   : Dining & Entertaining
title   : Artistic Churchware Communion Cup Filler: RW525
cleaned : artistic churchware communion cup filler rw525
features: {'Product_Type': 'cup'}

cat_3   : Bathroom Accessories
title   : 4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING
cleaned : 4 bar mysore sandal soap 70grams fast shipping
features: {}

cat_3   : Home Fragrance
title   : AROGYA VATI (40gm) by popeye seller
cleaned : arogya vati 40gm popeye seller
features: {'Capacity_Volume': '40gm'}



## 3. Run the extraction

`run_feature_extraction` chains every stage. Pass paths instead of pre-loaded objects to verify the JSON-loading branch works too.

In [22]:
# Edit this list to add or remove text columns to extract from.
# `list_columns` is the subset whose raw values are stringified lists.
# Takes ~40min when running on Apple M4 chip
text_columns = ['title', 'description', 'feature']
list_columns = ['description', 'feature']

df_result = run_feature_extraction(
    df_items,
    text_columns=text_columns,
    master_metadata=str(DATA_DIR / 'master_metadata.json'),
    global_filters=str(DATA_DIR / 'global_filters.json'),
    list_columns=list_columns,
    priority=text_columns,   # highest-to-lowest; defaults to text_columns order
)

print(f'Result shape: {df_result.shape}')
print(f'\nNew columns:')
expected = (
    [f'cat_{i+1}' for i in range(6)]
    + [f'{c}_cleaned' for c in text_columns]
    + [f'extracted_features_{c}' for c in text_columns]
    + ['extracted_features']
)
for col in expected:
    print(f'  {col:35s} present: {col in df_result.columns}')

Result shape: (1134566, 27)

New columns:
  cat_1                               present: True
  cat_2                               present: True
  cat_3                               present: True
  cat_4                               present: True
  cat_5                               present: True
  cat_6                               present: True
  title_cleaned                       present: True
  description_cleaned                 present: True
  feature_cleaned                     present: True
  extracted_features_title            present: True
  extracted_features_description      present: True
  extracted_features_feature          present: True
  extracted_features                  present: True


In [23]:
# Per-source vs combined coverage
from collections import Counter

total = len(df_result)
for src in ['title', 'description', 'feature']:
    n = (df_result[f'extracted_features_{src}'].apply(len) > 0).sum()
    print(f'  {src:12s}: {n:,} / {total:,} ({n/total*100:.1f}%)')

n_combined = (df_result['extracted_features'].apply(len) > 0).sum()
print(f'  combined    : {n_combined:,} / {total:,} ({n_combined/total*100:.1f}%)')

field_counts = Counter()
for d in df_result['extracted_features']:
    field_counts.update(d.keys())
print('\nTop fields extracted (combined):')
for field, count in field_counts.most_common(10):
    print(f'  {field:20s} {count:,} ({count/total*100:.1f}%)')

  title       : 1,048,373 / 1,134,566 (92.4%)
  description : 915,592 / 1,134,566 (80.7%)
  feature     : 875,765 / 1,134,566 (77.2%)
  combined    : 1,112,629 / 1,134,566 (98.1%)

Top fields extracted (combined):
  Product_Type         838,262 (73.9%)
  Material             761,739 (67.1%)
  Features             638,614 (56.3%)
  Dimensions           591,890 (52.2%)
  Color                530,245 (46.7%)
  Piece_Count          251,125 (22.1%)
  Theme                181,559 (16.0%)
  Brand                152,854 (13.5%)
  Capacity_Volume      114,224 (10.1%)
  Size                 81,941 (7.2%)


## 4. Expand to per-field columns

Turn the merged `extracted_features` dict into typed columns: one column per field (`Color`, `Material`, `Dimensions`, …), `<field>_numeric` + `<field>_unit` for capacity/weight/etc., `dimension_1/2/3/_unit` for dimensions, and standardized units via `UNIT_MAP`. This is what produces the table you saw in `create_features.ipynb`.

In [29]:
from feature_extraction_workflow import expand_features

df_expanded = expand_features(df_result)

print(f'Expanded shape: {df_expanded.shape}')

# Show numeric/unit columns produced
numeric_cols = [c for c in df_expanded.columns if c.endswith('_numeric')]
unit_cols = [c for c in df_expanded.columns if c.endswith('_unit')]
dim_cols = [c for c in df_expanded.columns if c.startswith('dimension_')]

print(f'\nNumeric columns ({len(numeric_cols)}):')
for c in numeric_cols:
    n = df_expanded[c].notna().sum()
    print(f'  {c:30s} {n:,} non-null')

print(f'\nUnit columns ({len(unit_cols)}):')
for c in unit_cols:
    units = df_expanded[c].dropna().unique()
    print(f'  {c:30s} units: {sorted(units)[:10]}')

print(f'\nDimension columns:')
for c in dim_cols:
    n = df_expanded[c].notna().sum()
    print(f'  {c:20s} {n:,} non-null')

Expanded shape: (1134566, 71)

Numeric columns (8):
  capacity_volume_numeric        114,224 non-null
  piece_count_numeric            157,366 non-null
  thread_count_numeric           19,346 non-null
  weight_numeric                 252 non-null
  bar_pressure_numeric           460 non-null
  capacity_cups_numeric          2,701 non-null
  stage_count_numeric            362 non-null
  voltage_numeric                2,363 non-null

Unit columns (5):
  capacity_volume_unit           units: ['bottle', 'cubic foot', 'cup', 'fl oz', 'g', 'gal', 'l', 'lb', 'ml', 'oz']
  piece_count_unit               units: ['bottle', 'capacity', 'chair', 'cone', 'count', 'door', 'drawer', 'dz', 'hook', 'in 1']
  thread_count_unit              units: ['count', 'series', 'thread count']
  weight_unit                    units: ['g', 'lb']
  dimension_unit                 units: ['cm', 'cm - m', 'cm --m', 'cm cm', 'cm in', 'cm m', 'cm meter', 'count', 'count foot', 'count m']

Dimension columns:
  dimension_1 

## 5. Clean categorical values, brands and dimensions

`master_metadata.json` lists some spelling variants as separate vocabulary
entries (`grey` and `gray`, `nonstick` and `non-stick`), so the extractor stores
whichever one the listing used and the same concept ends up under two labels.

`canonicalize_values` folds each variant onto one label using the fixed
`CANONICAL_VALUES` map. Both spellings stay in the schema — deleting one would
stop it matching — and the merge happens here instead. Unlike the range filter
this replaces in place: the two spellings mean the same thing, so there is
nothing to compare against.

`clean_brand` then handles the `brand` column. It isn't extracted — it comes
straight from the source metadata — but it arrives with ~98k distinct values,
most carried by a couple of items, so it is folded here rather than separately
in every downstream notebook. Spellings are merged first (`3d rose` / `3drose`),
then brands on 10 or fewer distinct items become `other_brands`. The original
`brand` column is kept, so a different threshold can be recomputed without
re-extracting.

Finally `normalize_dimensions` puts the three dimension columns on one scale.
It resolves the unit (`inch round` → `inch`, `cm m` → `cm`), converts every
recognised length to **inches** in new `dimension_1_in` / `_2_in` / `_3_in`
columns, and rewrites `dimension_unit_clean` so converted rows read `in` and a
unit that was present but unrecognised (`tall`, `w`, `oz`, `count`) reads
`Other`. A missing unit stays missing. It must run before the range filter,
which carries `0 < v < 151` bounds for the new columns.

In [ ]:
from feature_extraction_workflow import canonicalize_values, CANONICAL_VALUES

before = {f: df_expanded[f].nunique() for f in CANONICAL_VALUES if f in df_expanded.columns}
df_canon = canonicalize_values(df_expanded)

print('Categorical values folded onto one label:\n')
for f, n_before in before.items():
    n_after = df_canon[f].nunique()
    print(f'  {f:14s} {n_before:,} -> {n_after:,} distinct '
          f'({n_before - n_after} merged, {len(CANONICAL_VALUES[f])} aliases defined)')

untouched = [f for f in CANONICAL_VALUES if f not in df_expanded.columns]
if untouched:
    print(f'\n  not present in df: {untouched}')

from feature_extraction_workflow import clean_brand, BRAND_MIN_ITEMS, BRAND_OTHER

df_canon = clean_brand(df_canon)
n0, n1 = df_canon['brand'].nunique(), df_canon['brand_clean'].nunique()
print(f'\nbrand: {n0:,} -> {n1:,} labels '
      f'(> {BRAND_MIN_ITEMS} items kept, rest -> {BRAND_OTHER!r})')
print(f"  items in {BRAND_OTHER}: {(df_canon['brand_clean'] == BRAND_OTHER).mean():.1%}")
print(f'  items with no brand   : {df_canon["brand_clean"].isna().mean():.1%} (left as NaN)')

from feature_extraction_workflow import normalize_dimensions

df_canon = normalize_dimensions(df_canon)
uc = df_canon['dimension_unit_clean']
print(f"\ndimension_unit: {df_canon['dimension_unit'].nunique():,} distinct "
      f"-> {uc.nunique():,} after conversion")
print(uc.value_counts(dropna=False).head(5).to_string())
for c in ('dimension_1_in', 'dimension_2_in', 'dimension_3_in'):
    print(f"  {c}: {df_canon[c].notna().sum():,} ({df_canon[c].notna().mean():.1%})")

## 6. Clean numeric features to valid ranges

`clean_numeric_ranges` keeps the original columns intact and adds `<col>_cleaned` versions where out-of-range values are set to NaN. Same range table as `notebooks/analyze_features.ipynb`.

In [ ]:
from feature_extraction_workflow import clean_numeric_ranges, VALID_RANGES

df_clean = clean_numeric_ranges(df_canon)

print('Range filter applied — values outside the interval set to NaN:\n')
for col, bounds in VALID_RANGES.items():
    low, high = bounds[0], bounds[1]
    inc = bounds[2] if len(bounds) > 2 else 'both'
    lo_b = '[' if inc in ('both', 'left') else '('
    hi_b = ']' if inc in ('both', 'right') else ')'
    if col not in df_clean.columns:
        print(f'  {col}: not present in df, skipped')
        continue
    before = df_clean[col].notna().sum()
    after = df_clean[f'{col}_cleaned'].notna().sum()
    removed = before - after
    pct = (removed / before * 100) if before else 0.0
    print(f'  {col} {lo_b}{low}, {high}{hi_b}: {before:,} -> {after:,} ({removed:,} dropped, {pct:.1f}% of non-null)')

## 7. Save the final dataframe

Persist `df_clean` (cleaned text + per-source dicts + merged dict + expanded fields + numeric/unit + dimensions + `_cleaned` numerics) to `data/df_features.pkl` so the embedding-analysis notebook can pick it up directly.

In [34]:
from feature_extraction_workflow import save_features

out_path = DATA_DIR / 'df_features.pkl'
save_features(df_clean, out_path)

print(f'Saved {len(df_clean):,} rows x {df_clean.shape[1]} cols -> {out_path}')
print(f'\nIncluded columns include:')
print(f'  product id : asin')
print(f'  text       : ' + ', '.join(c for c in ["title", "description", "feature"] if c in df_clean.columns))
print(f'  cleaned txt: ' + ', '.join(c for c in df_clean.columns if c.endswith("_cleaned") and c.split("_")[0] in ["title", "description", "feature"]))
print(f'  per-source : ' + ', '.join(c for c in df_clean.columns if c.startswith("extracted_features_")))
print(f'  merged     : extracted_features')
print(f'  field cols : ' + str(len([c for c in df_clean.columns if c[:1].isupper()])) + ' columns (Brand, Color, Material, ...)')
print(f'  numerics   : ' + ', '.join(c for c in df_clean.columns if c.endswith("_numeric") or c.endswith("_w") or c.endswith("_lb") or c.endswith("_in")))
print(f'  numerics_cl: ' + ', '.join(c for c in df_clean.columns if c.endswith("_cleaned") and c not in ["title_cleaned", "description_cleaned", "feature_cleaned"]))

Saved 1,134,566 rows x 82 cols -> /Users/lazr/PycharmProjects/RecSystem/data/df_features.pkl

Included columns include:
  product id : asin
  text       : title, description, feature
  cleaned txt: title_cleaned, description_cleaned, feature_cleaned
  per-source : extracted_features_title, extracted_features_description, extracted_features_feature
  merged     : extracted_features
  field cols : 25 columns (Brand, Color, Material, ...)
  numerics   : capacity_volume_numeric, piece_count_numeric, thread_count_numeric, weight_numeric, bar_pressure_numeric, capacity_cups_numeric, density_weight_lb, pocket_depth_in, power_rating_w, stage_count_numeric, voltage_numeric
  numerics_cl: bar_pressure_numeric_cleaned, capacity_cups_numeric_cleaned, density_weight_lb_cleaned, pocket_depth_in_cleaned, power_rating_w_cleaned, stage_count_numeric_cleaned, voltage_numeric_cleaned, capacity_volume_numeric_cleaned, weight_numeric_cleaned, piece_count_numeric_cleaned, thread_count_numeric_cleaned


---

Everything above is a thin wrapper. To reproduce the pickle without the
commentary, the pipeline is:

```python
from feature_extraction_workflow import (
    run_feature_extraction, expand_features, canonicalize_values,
    clean_brand, normalize_dimensions, clean_numeric_ranges, save_features,
)

df = run_feature_extraction(
    df_items,
    text_columns=['title', 'description', 'feature'],
    master_metadata=str(DATA_DIR / 'master_metadata.json'),
    global_filters=str(DATA_DIR / 'global_filters.json'),
    list_columns=['description', 'feature'],
    priority=['title', 'description', 'feature'],
)
df = clean_numeric_ranges(normalize_dimensions(clean_brand(
        canonicalize_values(expand_features(df)))))
save_features(df, DATA_DIR / 'df_features.pkl')
```